In [3]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from polysim import extrusion, OUTPUTS

# the first import compiles LEF_Dynamics.pyx via pyximport -- a few seconds, once
print("ready")

ready


In [ ]:
# --- chain and LEFs ---
NPOLY   = 70000        # monomers
SEP     = 240          # monomers per LEF -> NLEFS = NPOLY // SEP
LIFE    = 75000        # LEF lifetime, in LEF timesteps
VLEF    = 0.005        # p(step per leg per timestep)

# --- CTCF ---
STALL      = 0.8       # stall probability per encounter
CTCF_LEFT  = extrusion.tile_sites([200, 330, 724, 1425, 1433, 1604], period=2000, length=NPOLY)  # right pointing
CTCF_RIGHT = extrusion.tile_sites([574, 694, 866, 1241, 1390, 1580, 1752, 1800], period=2000, length=NPOLY)  # left pointing
LIFEBOOSTSTALLED = 4   # lifetime multiplier while stalled at a CTCF (1 = no boost)

# --- schedule ---
INITSTEPS = 2_000_000  # equilibration steps, discarded
NUMSAVE   = 7200       # recorded frames
SAVEEVERY = 3000       # LEF steps between frames



291 LEFs, recording 7200 frames over 21600000 LEF steps (288.0 lifetimes)


In [ ]:
OUTDIR = "/mnt/md1/jjusuf/polysim/outputs/sweep1D"
os.makedirs(OUTDIR, exist_ok=True)

def run_1d_sim(NPOLY, SEP, LIFE, VLEF, STALL, CTCF_LEFT, CTCF_RIGHT, LIFEBOOSTSTALLED, INITSTEPS, NUMSAVE, SAVEEVERY):

    NLEFS = NPOLY // SEP

    stall_left, stall_right = extrusion.build_stall_arrays(
        NPOLY, ctcf_left=CTCF_LEFT, ctcf_right=CTCF_RIGHT, stall_prob=STALL)

    arrays = extrusion.build_lef_arrays(
        NPOLY, lifetime=LIFE, vlef=VLEF,
        stall_left=stall_left, stall_right=stall_right,
        life_boost_stalled=LIFEBOOSTSTALLED)

    smc = extrusion.make_translocator(arrays, NLEFS)

    n_sites = int(((stall_left > 0) | (stall_right > 0)).sum())

    smc.steps(INITSTEPS)

    positions = np.zeros((NUMSAVE, NLEFS, 2), dtype=np.int64)
    for i in tqdm(range(NUMSAVE)):
        smc.steps(SAVEEVERY)
        left, right = smc.getLEFs()
        positions[i, :, 0] = left
        positions[i, :, 1] = right

    np.save(os.path.join(OUTDIR, "LEFpositions.npy"), positions)
    np.savez(os.path.join(OUTDIR, "sites.npz"), **arrays)
    print("saved to", os.path.abspath(OUTDIR))
